<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Ejercicios/03_Taller_BoW_NGramas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Taller práctico: Bag-of-Words y N-gramas</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller practica los temas de `06_BoW_NGramas.ipynb` y `07_Clasificacion_NLTK.ipynb` con **datos y ejercicios distintos** a los de las notas de clase. En lugar de noticias en español, usaremos **novelas en inglés** del corpus Gutenberg de NLTK, de tres autores: Jane Austen, G. K. Chesterton y Herman Melville.

Enfocamos:

- la dispersión de los N-gramas y el crecimiento del vocabulario,
- la elección de hiperparámetros de suavizado **con un conjunto de validación**,
- la interpolación **bien normalizada** de unigramas, bigramas y trigramas,
- el autocompletado y la generación con muestreo *top-k*,
- la **atribución de autoría** con modelos de lenguaje (clasificación generativa) frente a Naive Bayes.

Resuelve cada ejercicio en la celda indicada. Cuando se pida una interpretación, respóndela en una celda markdown.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Ejercicio 1. Dispersión y crecimiento del vocabulario</span>](#ejercicio-1)
- [<span style="color: #005C99;">Ejercicio 2. Modelo de bigramas con add-k y validación</span>](#ejercicio-2)
- [<span style="color: #005C99;">Ejercicio 3. Interpolación normalizada de trigramas</span>](#ejercicio-3)
- [<span style="color: #005C99;">Ejercicio 4. Autocompletado y generación con muestreo top-k</span>](#ejercicio-4)
- [<span style="color: #005C99;">Ejercicio 5. Atribución de autoría</span>](#ejercicio-5)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib nltk

In [ ]:
import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import gutenberg
from nltk.classify import NaiveBayesClassifier, accuracy

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

_ = nltk.download("gutenberg", quiet=True)

LIBROS = {
    "austen": ["austen-emma.txt", "austen-persuasion.txt", "austen-sense.txt"],
    "chesterton": ["chesterton-ball.txt", "chesterton-brown.txt", "chesterton-thursday.txt"],
    "melville": ["melville-moby_dick.txt"],
}


def limpiar(oracion):
    """Minúsculas y solo tokens con alguna letra o signos de puntuación básicos."""
    return [w.lower() for w in oracion if any(c.isalpha() for c in w) or w in {".", ",", ";", "!", "?"}]


oraciones = {autor: [s for f in archivos for s in map(limpiar, gutenberg.sents(f)) if len(s) >= 3]
             for autor, archivos in LIBROS.items()}

# División 80/10/10 por autor (entrenamiento / validación / prueba)
particion = {}
for autor, sents in oraciones.items():
    sents = sents[:]
    random.Random(SEED).shuffle(sents)
    n = len(sents)
    particion[autor] = {"train": sents[:int(0.8 * n)], "dev": sents[int(0.8 * n):int(0.9 * n)],
                        "test": sents[int(0.9 * n):]}
    print(f"{autor:<11} {n:>6} oraciones")

---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Dispersión y crecimiento del vocabulario</h2>

**Objetivo:** cuantificar cómo crece el número de N-gramas distintos con el tamaño del corpus y con $N$.

1. Implementa `ngramas(tokens, n)`.
2. Con las oraciones de **entrenamiento de Austen**, calcula para $N = 1, \dots, 4$ el número de N-gramas distintos y el porcentaje que aparece una sola vez. Presenta una tabla.
3. Grafica la **curva de crecimiento del vocabulario**: en el eje X el número de tokens leídos (recorriendo las oraciones en orden) y en el eje Y el número de unigramas distintos y de bigramas distintos vistos hasta ese momento. Usa escala log-log.
4. La **ley de Heaps** dice que el vocabulario crece como $V(n) \approx K n^{\beta}$ con $\beta < 1$. Estima $\beta$ para unigramas y bigramas ajustando una recta en escala log-log. ¿Qué significa que el vocabulario de bigramas crezca más rápido? ¿Qué implica para un modelo de trigramas?

In [ ]:
# Escribe tu código aquí
def ngramas(tokens, n):
    pass

---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Modelo de bigramas con add-k y validación</h2>

**Objetivo:** implementar un modelo de bigramas con vocabulario cerrado y elegir $k$ con un conjunto de validación.

1. Construye el vocabulario `VOCAB` con las palabras que aparecen al menos 2 veces en el **entrenamiento de Austen**, más `<UNK>` y `</s>`. Implementa `unk(oracion)` que reemplace las palabras fuera del vocabulario por `<UNK>`.
2. Implementa la clase `BigramaAddK` con los métodos `entrenar(oraciones)`, `prob(palabra, previa)` y `log_prob(oracion)` (usa `<s>` al inicio y `</s>` al final).
3. Implementa `perplejidad(modelo, oraciones)`.
4. Evalúa $k \in \{1, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001\}$ sobre el conjunto de **validación** de Austen y grafica la perplejidad contra $k$ (escala log en X).
5. Con el mejor $k$, reporta la perplejidad en **prueba**. ¿Por qué no elegimos $k$ directamente con el conjunto de prueba?

In [ ]:
# Escribe tu código aquí
class BigramaAddK:
    def __init__(self, k):
        self.k = k

    def entrenar(self, oraciones):
        pass

    def prob(self, palabra, previa):
        pass

    def log_prob(self, oracion):
        pass


def perplejidad(modelo, oraciones):
    pass

---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Interpolación normalizada de trigramas</h2>

**Objetivo:** implementar una interpolación lineal que siempre sea una distribución de probabilidad válida y ajustar sus pesos en validación.

En el cuaderno 06, cuando un contexto no se había visto, su término de la interpolación simplemente no aportaba, así que las "probabilidades" no sumaban 1. Aquí lo haremos bien con una definición **recursiva**:

$$P_1(w) = \frac{C(w) + 1}{N + |V|}$$

$$P_2(w \mid u) = \begin{cases} \lambda_2 \, P_{\text{MLE}}(w \mid u) + (1 - \lambda_2)\, P_1(w) & \text{si } C(u) > 0 \\ P_1(w) & \text{si no} \end{cases}$$

$$P_3(w \mid u, v) = \begin{cases} \lambda_3 \, P_{\text{MLE}}(w \mid u, v) + (1 - \lambda_3)\, P_2(w \mid v) & \text{si } C(u, v) > 0 \\ P_2(w \mid v) & \text{si no} \end{cases}$$

1. Implementa la clase `TrigramaInterpolado(l2, l3)` con `entrenar`, `prob(palabra, u, v)` y `log_prob(oracion)` (usa dos `<s>` al inicio).
2. Comprueba numéricamente que, para un contexto cualquiera, $\sum_{w \in V} P_3(w \mid u, v) = 1$ (sin contar `<s>`, que nunca se predice).
3. Busca en una grilla $\lambda_2, \lambda_3 \in \{0.1, 0.3, 0.5, 0.7, 0.9\}$ la combinación con menor perplejidad en **validación**.
4. Compara en **prueba**: unigrama ($P_1$), bigrama add-k (ejercicio 2), bigrama interpolado ($\lambda_3 = 0$) y trigrama interpolado.

In [ ]:
# Escribe tu código aquí
class TrigramaInterpolado:
    def __init__(self, l2, l3):
        self.l2, self.l3 = l2, l3

    def entrenar(self, oraciones):
        pass

    def prob(self, palabra, u, v):
        pass

    def log_prob(self, oracion):
        pass

---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Autocompletado y generación con muestreo top-k</h2>

**Objetivo:** usar el modelo de lenguaje para sugerir palabras y generar texto, controlando la diversidad.

1. Implementa `sugerir(prefijo, k=5)` que, dado un texto (por ejemplo `"she was very"`), devuelva las $k$ palabras más probables como continuación según el trigrama interpolado del ejercicio 3, con sus probabilidades. Excluye `<UNK>` de las sugerencias.
2. Pruébalo con al menos cuatro prefijos.
3. Implementa `generar(modelo, top_k, max_palabras=25)`: en cada paso calcula la distribución sobre todo el vocabulario, conserva solo las `top_k` palabras más probables (sin `<UNK>`), renormaliza y muestrea. Termina al generar `</s>`.
4. Genera tres oraciones con `top_k=1` (decodificación voraz), `top_k=10` y `top_k=200`. Describe las diferencias.

In [ ]:
# Escribe tu código aquí
def sugerir(prefijo, k=5):
    pass


def generar(modelo, top_k, max_palabras=25, rng=None):
    pass

---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Atribución de autoría</h2>

**Objetivo:** clasificar oraciones por autor con modelos de lenguaje (un clasificador generativo) y compararlo con Naive Bayes de NLTK.

Un modelo de lenguaje por clase es un **clasificador generativo**: asignamos la oración al autor cuyo modelo le da mayor probabilidad (con priors iguales):

$$\hat{a} = \arg\max_{a} P_a(\text{oración})$$

1. Construye un vocabulario **común** con las palabras que aparecen al menos 2 veces en el entrenamiento de **todos** los autores juntos (así las probabilidades de los distintos modelos son comparables).
2. Entrena un `TrigramaInterpolado` por autor (puedes usar $\lambda_2 = 0.7$, $\lambda_3 = 0.5$ o los mejores del ejercicio 3).
3. Clasifica las oraciones de **prueba** de los tres autores y reporta la exactitud global y la matriz de confusión.
4. Entrena un `NaiveBayesClassifier` de NLTK con características `contiene(palabra)` para las 2 000 palabras más frecuentes y compáralo.
5. Agrupa las oraciones de prueba por longitud (por ejemplo: 3–8, 9–15, 16–30, más de 30 tokens) y grafica la exactitud de ambos métodos por grupo. Interpreta.

In [ ]:
# Escribe tu código aquí

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Bird, S., Klein, E. & Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media. [nltk.org/book](https://www.nltk.org/book/)
- Heaps, H. S. (1978). *Information Retrieval: Computational and Theoretical Aspects*. Academic Press.
- Stamatatos, E. (2009). A survey of modern authorship attribution methods. *JASIST*, 60(3).